[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/12_gradient_boosting/90_exercises.ipynb)

# Bài tập — Chương 12: Gradient Boosting (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/12_gradient_boosting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import log_loss, mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d, make_regression_1d
from ml_course.utils import repo_path, sigmoid

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết: công thức cập nhật $\alpha_m$ và trọng số mẫu của AdaBoost; bước cập nhật của Gradient Boosting; pseudo-residual cho MSE và log-loss; giá trị lá tối ưu và gain của XGBoost.

✍️ Trả lời:

**Đáp án:** AdaBoost: $\alpha_m = \frac12\ln\frac{1 - \varepsilon_m}{\varepsilon_m}$, $w_i \leftarrow w_i e^{-\alpha_m y_i h_m(x_i)}$ rồi chuẩn hoá. GB: $F_m = F_{m-1} + \eta h_m$, $h_m$ fit vào $r_i = -\partial L/\partial F$. MSE: $r = y - F$; log-loss: $r = y - \sigma(F)$. XGBoost: $w^* = -\frac{G}{H + \lambda}$; $\text{Gain} = \frac12\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma$.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích bằng ví dụ đời thường vì sao learning rate nhỏ + nhiều cây thường tốt hơn learning rate lớn + ít cây, và vì sao Gradient Boosting cần early stopping còn Random Forest thì không.

✍️ Trả lời:

**Gợi ý:** như chỉnh âm lượng: vặn mạnh (η lớn) dễ "quá tay" và phải sửa ngược nhiều lần, bám theo cả tiếng ồn; vặn từng nấc nhỏ chậm hơn nhưng dừng đúng chỗ — mỗi cây chỉ sửa một phần nhỏ lỗi nên các cây sau có cơ hội sửa chung, ít phụ thuộc vào một cây "nhiễu". Boosting **cộng dồn** sửa lỗi: càng nhiều vòng model càng phức tạp, cuối cùng học thuộc nhiễu → cần dừng khi validation không cải thiện. Random Forest **lấy trung bình** các cây độc lập: thêm cây chỉ làm trung bình ổn định hơn, không tăng độ phức tạp.

## L2 — Tính tay

**Câu 3.** AdaBoost: stump đầu tiên có lỗi có trọng số $\varepsilon_1 = 0.2$ trên 10 mẫu (trọng số đều). Tính $\alpha_1$, trọng số mới (chưa chuẩn hoá và đã chuẩn hoá) của một mẫu bị sai và một mẫu đúng.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
# STARTER: alpha1 = None
# STARTER: w_wrong_raw = None      # 0.1 · e^{α}
# STARTER: w_right_raw = None      # 0.1 · e^{−α}
# STARTER: Z = None                # tổng trọng số mới (2 mẫu sai, 8 mẫu đúng)
# STARTER: w_wrong, w_right = None, None
alpha1 = 0.5 * np.log(0.8 / 0.2)
w_wrong_raw = 0.1 * np.exp(alpha1)
w_right_raw = 0.1 * np.exp(-alpha1)
Z = 2 * w_wrong_raw + 8 * w_right_raw
w_wrong, w_right = w_wrong_raw / Z, w_right_raw / Z
### END CODE HERE ###

In [ ]:
check_close(alpha1, np.log(2), message="α₁ = ½ ln 4 = ln 2")
check_close((w_wrong_raw, w_right_raw), (0.2, 0.05), message="trọng số chưa chuẩn hoá")
check_close((w_wrong, w_right), (0.25, 0.0625), message="sau chuẩn hoá: 2 mẫu sai chiếm 50% tổng trọng số")

**Câu 4.** GB hồi quy (MSE), $y = [3, 5, 10]$, $F_0$ = trung bình. Cây đầu tiên (stump) tách mẫu 1–2 | mẫu 3 và dự đoán trung bình residual ở mỗi lá. Với $\eta = 0.5$, tính $F_0$, residual, giá trị lá, $F_1$ và MSE sau bước 1.

In [ ]:
y_toy = np.array([3.0, 5.0, 10.0])
### START CODE HERE ### (≈ 5 dòng)
# STARTER: F0 = None
# STARTER: residual = None
# STARTER: leaf_left, leaf_right = None, None     # trung bình residual ở mỗi lá
# STARTER: F1 = None
# STARTER: mse1 = None
F0 = y_toy.mean()
residual = y_toy - F0
leaf_left, leaf_right = residual[:2].mean(), residual[2]
F1 = F0 + 0.5 * np.array([leaf_left, leaf_left, leaf_right])
mse1 = np.mean((y_toy - F1) ** 2)
### END CODE HERE ###

In [ ]:
check_close(F0, 6.0, message="F0")
check_close(residual, [-3, -1, 4], message="residual")
check_close((leaf_left, leaf_right), (-2.0, 4.0), message="giá trị lá")
check_close(F1, [5, 5, 8], message="F1")
check_close(mse1, (4 + 0 + 4) / 3, message="MSE sau bước 1")

## L3 — Code

**Bài 5.** Cài đặt `gb_fit_huber(X, y, n_estimators, lr, max_depth, delta)`: Gradient Boosting với **Huber loss** — pseudo-residual $r = y - F$ nếu $|y - F| \le \delta$, ngược lại $\delta\cdot\operatorname{sign}(y - F)$. Khởi tạo $F_0$ = trung vị. (Giá trị lá = trung bình pseudo-residual — bản đơn giản.) Trả về `(F0, trees)`; viết `gb_predict`.

In [ ]:
def gb_fit_huber(X, y, n_estimators=200, lr=0.1, max_depth=2, delta=1.0):
    F0 = np.median(y)
    F = np.full(len(y), F0)
    trees = []
    for _ in range(n_estimators):
        ### START CODE HERE ### (≈ 5 dòng)
        # STARTER: diff = None
        # STARTER: r = None          # pseudo-residual Huber
        # STARTER: tree = None       # DecisionTreeRegressor(max_depth=max_depth, random_state=0) fit vào r
        # STARTER: # cập nhật F += lr * tree.predict(X); thêm tree vào trees
        diff = y - F
        r = np.where(np.abs(diff) <= delta, diff, delta * np.sign(diff))
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, r)
        F += lr * tree.predict(X)
        trees.append(tree)
        ### END CODE HERE ###
    return F0, trees


def gb_predict(F0, trees, X, lr=0.1):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return None
    return F0 + lr * sum(t.predict(X) for t in trees)
    ### END CODE HERE ###

In [ ]:
Xr, yr = make_regression_1d(n=300, kind="sin", noise=0.3, seed=2)
Xr_val, yr_val = make_regression_1d(n=1000, kind="sin", noise=0.3, seed=3)
yr_dirty = yr.copy()
yr_dirty[rng.choice(300, 25, replace=False)] += 15                     # 8% outlier một phía
F0_h, trees_h = gb_fit_huber(Xr, yr_dirty, delta=1.0)
F0_m, trees_m = gb_fit_huber(Xr, yr_dirty, delta=1e9)                  # delta rất lớn = MSE (nhưng F0 = trung vị)
mse_huber = mean_squared_error(yr_val, gb_predict(F0_h, trees_h, Xr_val))
mse_plain = mean_squared_error(yr_val, gb_predict(F0_m, trees_m, Xr_val))
print(f"MSE validation: Huber = {mse_huber:.3f} | MSE-loss = {mse_plain:.3f}")
check_less(mse_huber, 0.5 * mse_plain, "Huber robust hơn hẳn với outlier")

**Bài 6.** Cài đặt `xgb_best_split(x, g, h, lam, gamma)` — với một feature `x`, gradient `g`, hessian `h`: thử mọi ngưỡng (trung điểm giữa hai giá trị liên tiếp khác nhau sau khi sắp xếp), trả về `(threshold, gain)` có gain lớn nhất theo công thức XGBoost. Dùng `cumsum` để có $G_L, H_L$ cho mọi ngưỡng.

In [ ]:
def xgb_best_split(x, g, h, lam=1.0, gamma=0.0):
    order = np.argsort(x)
    xs, gs, hs = x[order], g[order], h[order]
    G, H = gs.sum(), hs.sum()
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: GL, HL = None, None     # tổng tích luỹ gradient/hessian bên trái tại mỗi vị trí cắt
    # STARTER: GR, HR = None, None
    # STARTER: gain = None             # công thức gain XGBoost; đặt -inf tại vị trí xs[i] == xs[i+1]
    # STARTER: i = None
    # STARTER: threshold = None
    # STARTER: best_gain = None
    GL, HL = np.cumsum(gs)[:-1], np.cumsum(hs)[:-1]
    GR, HR = G - GL, H - HL
    gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G**2 / (H + lam)) - gamma
    gain[xs[1:] == xs[:-1]] = -np.inf
    i = int(np.argmax(gain))
    threshold = (xs[i] + xs[i + 1]) / 2
    best_gain = gain[i]
    ### END CODE HERE ###
    return threshold, best_gain

In [ ]:
y_toy6 = np.array([1, 1, 1, 0, 0, 1])
x_toy6 = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
p6 = np.full(6, 0.5)
t6, g6 = xgb_best_split(x_toy6, p6 - y_toy6, p6 * (1 - p6), lam=1.0)
check_close(t6, 3.5, message="ngưỡng tốt nhất (ví dụ bài 03)")
check_close(g6, 0.5 * (2.25 / 1.75 + 0.25 / 1.75 - 1 / 2.5), message="gain")

## L4 — Debug

**Bài 7.** GB phân loại dưới đây có **3 lỗi**: (1) khởi tạo $F_0 = \bar y$ (xác suất) thay vì **log-odds**; (2) pseudo-residual sai dấu ($p - y$); (3) dự đoán trả về $F$ chứ không phải $\sigma(F)$. Sửa trong `gb_binary_fixed`.

In [ ]:
def gb_binary_fixed(X, y, X_new, n_estimators=100, lr=0.1, max_depth=3):
    ### START CODE HERE ### (≈ 9 dòng)
    # STARTER: F0 = y.mean()                                       # 🐛
    # STARTER: F = np.full(len(y), F0)
    # STARTER: trees = []
    # STARTER: for _ in range(n_estimators):
    # STARTER:     residual = sigmoid(F) - y                          # 🐛
    # STARTER:     tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, residual)
    # STARTER:     F += lr * tree.predict(X)
    # STARTER:     trees.append(tree)
    # STARTER: return F0 + lr * sum(t.predict(X_new) for t in trees)   # 🐛
    F0 = np.log(y.mean() / (1 - y.mean()))
    F = np.full(len(y), F0)
    trees = []
    for _ in range(n_estimators):
        residual = y - sigmoid(F)
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, residual)
        F += lr * tree.predict(X)
        trees.append(tree)
    return sigmoid(F0 + lr * sum(t.predict(X_new) for t in trees))
    ### END CODE HERE ###

In [ ]:
Xb, yb = make_classification_2d("moons", n=800, noise=0.3, seed=7)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb.astype(float), test_size=0.3, random_state=0)
p_fixed = gb_binary_fixed(Xb_tr, yb_tr, Xb_te)
check_true(np.all((p_fixed >= 0) & (p_fixed <= 1)), "đầu ra là xác suất")
check_greater(roc_auc_score(yb_te, p_fixed), 0.95, "ROC-AUC > 0.95")
check_less(log_loss(yb_te, p_fixed), 0.30, "log-loss < 0.30")

## L5 — Phân tích

**Bài 8.** Với `HistGradientBoostingRegressor` trên dữ liệu `house_prices.csv` (target `price_billion_vnd`, cột số), khảo sát lưới `learning_rate` ∈ {0.3, 0.1, 0.03} × `max_iter` ∈ {50, 200, 800} (không early stopping). Lưu MAE validation vào DataFrame `table` (hàng = learning_rate, cột = max_iter). Nhận xét quan hệ learning rate ↔ số cây.

In [ ]:
houses = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
num = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
Xh_tr, Xh_val, yh_tr, yh_val = train_test_split(houses[num], houses["price_billion_vnd"], test_size=0.3, random_state=0)
table = pd.DataFrame(index=[0.3, 0.1, 0.03], columns=[50, 200, 800], dtype=float)
### START CODE HERE ### (≈ 4 dòng)
# STARTER: # với mỗi lr (hàng) và n_iter (cột): train HistGradientBoostingRegressor(learning_rate=lr, max_iter=n_iter,
# STARTER: #     early_stopping=False, random_state=0), ghi MAE validation vào table.loc[lr, n_iter]
for lr in table.index:
    for n_iter in table.columns:
        m = HistGradientBoostingRegressor(learning_rate=lr, max_iter=n_iter, early_stopping=False, random_state=0).fit(Xh_tr, yh_tr)
        table.loc[lr, n_iter] = mean_absolute_error(yh_val, m.predict(Xh_val))
### END CODE HERE ###
table.round(3)

In [ ]:
check_less(table.loc[0.03, 800], table.loc[0.03, 50], "η nhỏ cần nhiều cây hơn")
check_greater(table.loc[0.3, 800], table.values.min(), "η lớn + nhiều cây không phải cấu hình tốt nhất")

✍️ Nhận xét:

**Gợi ý:** với η = 0.03, 50 cây là quá ít (underfit) — cần hàng trăm cây; với η = 0.3, model tốt nhất đạt rất sớm và thêm cây làm tệ đi (overfit). Tích η × số cây gần như quyết định "lượng học"; cấu hình tốt nhất thường là η nhỏ–vừa với số cây được chọn bằng early stopping.

## L6 — Thực tế

**Bài 9.** Model churn tốt nhất có thể với `HistGradientBoostingClassifier` trên `customer_churn.csv`: dùng cột phân loại dạng `category` (`categorical_features="from_dtype"`), early stopping, tune `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization` bằng `RandomizedSearchCV` (≥ 10 cấu hình, 4-fold, `roc_auc`). Báo cáo ROC-AUC test (`test_auc`) — mục tiêu ≥ 0.80 — và so sánh với Logistic Regression baseline (`lr_auc`).

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = df.drop(columns=["customer_id", "churn"])
for c in ["contract", "internet_service", "payment_method"]:
    Xc[c] = Xc[c].astype("category")
Xc["senior"] = Xc["senior"].astype(int)
yc = df["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
### START CODE HERE ### (≈ 10 dòng)
search = RandomizedSearchCV(
    HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, categorical_features="from_dtype", early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=30, random_state=0),
    {"max_leaf_nodes": randint(4, 48), "min_samples_leaf": randint(10, 200), "l2_regularization": loguniform(1e-3, 10)},
    n_iter=12, cv=4, scoring="roc_auc", random_state=0).fit(Xc_tr, yc_tr)
test_auc = roc_auc_score(yc_te, search.predict_proba(Xc_te)[:, 1])
Xd_tr, Xd_te = pd.get_dummies(Xc_tr, dtype=float), pd.get_dummies(Xc_te, dtype=float)
lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xd_tr, yc_tr)
lr_auc = roc_auc_score(yc_te, lr_model.predict_proba(Xd_te)[:, 1])
### END CODE HERE ###
print(search.best_params_)
print(f"ROC-AUC test: HistGB = {test_auc:.4f} | Logistic Regression = {lr_auc:.4f}")

In [ ]:
check_greater(test_auc, 0.80, "ROC-AUC test ≥ 0.80")

✍️ Nhận xét: GB có thắng Logistic Regression nhiều không? Vì sao (gợi ý: dữ liệu churn được sinh từ một logit gần tuyến tính — xem `make_customer_churn`)?

**Gợi ý:** GB **không thắng** (thậm chí thua nhẹ) — vì quy luật sinh dữ liệu là log-odds **tuyến tính** theo các feature, Logistic Regression đã gần tối ưu, còn GB phải xấp xỉ quan hệ tuyến tính bằng các bậc thang. GB thắng đậm khi có tương tác/phi tuyến mạnh. Bài học: luôn có baseline đơn giản; model phức tạp chỉ đáng dùng khi lợi ích đủ lớn so với chi phí (độ trễ, diễn giải, bảo trì).

## L7 — Nâng cao

**Bài 10.** Cài đặt **XGBoost thu nhỏ** cho phân loại nhị phân: mỗi vòng tính $g = p - y$, $h = p(1 - p)$; xây **cây độ sâu 1** (stump) chọn split tốt nhất trên **mọi feature** bằng `xgb_best_split` (Bài 6), giá trị lá $w = -G/(H + \lambda)$; $F \leftarrow F + \eta\,w$. Train 200 vòng trên moons, so sánh log-loss test với `HistGradientBoostingClassifier(max_depth=1)` cùng learning rate.

In [ ]:
def mini_xgb(X, y, n_rounds=200, lr=0.3, lam=1.0):
    F = np.zeros(len(y))
    stumps = []                                       # (feature, threshold, w_left, w_right)
    for _ in range(n_rounds):
        p = sigmoid(F)
        g, h = p - y, p * (1 - p)
        ### START CODE HERE ### (≈ 9 dòng)
        # STARTER: best = None      # (feature, threshold, gain) tốt nhất qua mọi feature, dùng xgb_best_split
        # STARTER: left = None      # mask các mẫu bên trái split tốt nhất
        # STARTER: wl, wr = None, None   # giá trị lá -G/(H + lam)
        # STARTER: # F += lr * (wl nếu bên trái, wr nếu bên phải); stumps.append((j, t, wl, wr))
        best = None
        for j in range(X.shape[1]):
            t, gain = xgb_best_split(X[:, j], g, h, lam)
            if best is None or gain > best[2]:
                best = (j, t, gain)
        j, t, _ = best
        left = X[:, j] <= t
        wl, wr = -g[left].sum() / (h[left].sum() + lam), -g[~left].sum() / (h[~left].sum() + lam)
        F += lr * np.where(left, wl, wr)
        stumps.append((j, t, wl, wr))
        ### END CODE HERE ###
    return stumps


def mini_xgb_proba(stumps, X, lr=0.3):
    F = np.zeros(len(X))
    for j, t, wl, wr in stumps:
        F += lr * np.where(X[:, j] <= t, wl, wr)
    return sigmoid(F)

In [ ]:
stumps = mini_xgb(Xb_tr, yb_tr, n_rounds=200, lr=0.3)
p_mini = mini_xgb_proba(stumps, Xb_te)
hgb = HistGradientBoostingClassifier(max_depth=1, learning_rate=0.3, max_iter=200, early_stopping=False, random_state=0).fit(Xb_tr, yb_tr)
ll_mini, ll_hgb = log_loss(yb_te, p_mini), log_loss(yb_te, hgb.predict_proba(Xb_te)[:, 1])
print(f"log-loss test: mini XGBoost = {ll_mini:.4f} | HistGB (stump) = {ll_hgb:.4f}")
check_less(abs(ll_mini - ll_hgb), 0.03, "mini XGBoost ≈ HistGradientBoosting với stump")